# Optuna on UCI credit default — Random Forest and XGBoost

Predict **next-month default**. We do **not** try every tree setting. Optuna tries a few, remembers the score, and picks the next try.

Walk this notebook in order. Three Optuna steps, then we use them twice:

1. Write an **objective** (one trial = one model + one score).
2. **Create a study** (empty log of trials).
3. **`study.optimize(...)`** (run N trials; the study fills in).

Then `study.best_trial` is the best settings we found. We refit that model on all of train.

```bash
pip install -e ".[dev]"
```


## 1) Setup

`N_ROWS`, `N_TRIALS`, and `N_FOLDS` are only so this notebook finishes in minutes.

- Full file is ~30,000 rows. We use 2,000.
- Production would use more trials (see `config.N_TRIALS`) and 3 CV folds (`config.CV_FOLDS`).
- **More trials can still find a better set.** We are not stopping early.


In [15]:
import optuna
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

from credit_fraud import config
from credit_fraud.data import load_raw_dataframe, split_xy, train_test_split_stratified
from credit_fraud.features import build_features
from credit_fraud.modeling import (
    classification_metrics,
    make_random_forest,
    make_xgboost,
    predict_proba_positive,
    scale_pos_weight,
)

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Notebook speed only — not part of Optuna.
N_ROWS = 2000
N_TRIALS = 20
N_FOLDS = 2

print("target =", config.TARGET_COL)
print("notebook speed: rows / trials / folds =", N_ROWS, N_TRIALS, N_FOLDS)
print("production knobs: N_TRIALS =", config.N_TRIALS, " CV_FOLDS =", config.CV_FOLDS)


target = default_next_month
notebook speed: rows / trials / folds = 2000 20 2
production knobs: N_TRIALS = 15  CV_FOLDS = 3


## 2) Data

Same UCI credit-default file and features as the rest of the repo.

About **22%** of clients default. Always predicting “no default” is ~**78% accurate** and useless. We score **PR-AUC**. A random ranking sits near the default rate (**~0.22**). Higher is better.


In [16]:
raw = load_raw_dataframe()
x, y = split_xy(raw)
print(f"full data: {len(x):,} rows, default rate={y.mean():.1%}")

if len(x) > N_ROWS:
    x, _, y, _ = train_test_split(
        x, y, train_size=N_ROWS, stratify=y, random_state=config.RANDOM_STATE
    )
    x = x.reset_index(drop=True)
    y = y.reset_index(drop=True)

x_train, x_test, y_train, y_test = train_test_split_stratified(x, y)
x_train, x_test, _ = build_features(x_train, x_test, y_train)

dummy_acc = float((y_test == 0).mean())
baseline = float(y_train.mean())
print(f"train={len(x_train):,}  test={len(x_test):,}  features={x_train.shape[1]}")
print(
    f"default rate={y_train.mean():.1%}  "
    f"never-default accuracy≈{dummy_acc:.1%}  "
    f"random ranking PR-AUC≈{baseline:.2f}"
)


Loading local CSV: /Users/himanshujhamb/Desktop/Personal/Home_Assignment/Naresh/data/credit_default.csv
full data: 30,000 rows, default rate=22.1%
train=1,600  test=400  features=29
default rate=22.1%  never-default accuracy≈78.0%  random ranking PR-AUC≈0.22


## 3) The three Optuna steps (read this before the code)

### Step A — objective

A function Optuna calls **once per trial**. Inside it:

1. Ask Optuna for settings (`trial.suggest_int`, `trial.suggest_float`).
2. Build a model with those settings.
3. Return a number. We return **cross-validated PR-AUC** on train.

`suggest_int("max_depth", 4, 16)` means: give me an integer from 4 to 16, and remember it for this trial.

### Step B — create a study

A study is an **empty log**. Create it **before** you run trials.

- `direction="maximize"` — higher PR-AUC is better.
- **Sampler** = how the next settings are chosen.
  - **TPE** (Bayesian): first few trials are random, then it prefers settings that sat next to high scores.
  - **Random**: every trial is a fresh draw. No memory.

Creating the study does **not** train anything.

### Step C — `study.optimize(objective, n_trials=N)`

This **is** the search. Optuna:

1. Picks settings.
2. Calls `objective`.
3. Stores the score on the **same** `study` object (it is updated in place).
4. Repeats N times.

It does **not** stop when it finds a good trial. If the best score is trial 19 of 20, we still ran all 20. Trial 19 just had the highest number. Want a longer search? Raise `N_TRIALS`.

### After that

```python
study.best_trial.params   # the settings
study.best_trial.value    # the CV PR-AUC
```


### Shared helper: score one model with CV PR-AUC

Each trial trains on a few train/valid splits and averages PR-AUC. That number is what Optuna maximises. The hold-out test set is **not** used here — we save it for the refit at the end.


In [19]:
def pr_auc_scorer(estimator, x, y):
    return float(average_precision_score(y, predict_proba_positive(estimator, x)))


def cv_pr_auc(model, x, y):
    cv = StratifiedKFold(
        n_splits=N_FOLDS, shuffle=True, random_state=config.RANDOM_STATE
    )
    scores = cross_val_score(model, x, y, scoring=pr_auc_scorer, cv=cv, n_jobs=-1)
    return float(scores.mean())


## 4) Random Forest (TPE)

We tune three knobs:

| knob | meaning | range |
|------|---------|-------|
| `n_estimators` | how many trees | 100–400 |
| `max_depth` | how deep each tree may split | 4–16 |
| `min_samples_leaf` | min rows in a leaf (higher = simpler) | 1–20 |

### Step A — objective


In [20]:
def rf_objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 100, 400, step=50)
    max_depth = trial.suggest_int("max_depth", 4, 16)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
    model = make_random_forest(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
    )
    return cv_pr_auc(model, x_train, y_train)


### Step B — create the study

Empty log. TPE sampler. First 5 trials are random (`n_startup_trials=5`) so later trials have a history.


In [21]:
rf_study = optuna.create_study(
    direction="maximize",
    study_name="random_forest",
    sampler=optuna.samplers.TPESampler(seed=config.RANDOM_STATE, n_startup_trials=5),
)
print("trials so far:", len(rf_study.trials))


trials so far: 0


### Step C — run trials

`optimize` writes into `rf_study`. Same object before and after.


In [22]:
rf_study.optimize(rf_objective, n_trials=N_TRIALS)
print("trials after optimize:", len(rf_study.trials))


trials after optimize: 20


Each row is one trial. `pr_auc` is the CV score Optuna stored.


In [23]:
rf_rows = pd.DataFrame(
    [{"trial": t.number, "pr_auc": t.value, **t.params} for t in rf_study.trials]
)
rf_rows


,trial,pr_auc,n_estimators,max_depth,min_samples_leaf
0,0,0.496012,200,16,15
1,1,0.472496,300,6,4
2,2,0.486701,100,15,13
3,3,0.490840,300,4,20
4,4,0.473018,350,6,4
5,5,0.483758,100,15,12
6,6,0.489765,200,11,19
7,7,0.483205,400,11,10
8,8,0.439811,200,11,1
9,9,0.492942,200,16,16


Best settings = `study.best_trial`. That is how we pick the winner of the search.


In [24]:
print("best trial number:", rf_study.best_trial.number)
print("best params:", rf_study.best_trial.params)
print("best CV PR-AUC:", round(rf_study.best_trial.value, 3))
print(f"(random ranking ≈ {baseline:.2f} — we want well above that)")


best trial number: 0
best params: {'n_estimators': 200, 'max_depth': 16, 'min_samples_leaf': 15}
best CV PR-AUC: 0.496
(random ranking ≈ 0.22 — we want well above that)


Dots = that trial's score. Orange line = **best so far** (running max). It only goes up. If the best dot is near the end, that is fine — we still ran every trial.


In [25]:
numbers = [t.number for t in rf_study.trials]
scores = [t.value for t in rf_study.trials]
best_so_far = pd.Series(scores).cummax()

fig = go.Figure()
fig.add_trace(
    go.Scatter(x=numbers, y=scores, mode="markers+lines", name="this trial")
)
fig.add_trace(
    go.Scatter(
        x=numbers,
        y=best_so_far,
        mode="lines",
        name="best so far",
        line={"color": "#C45C26", "width": 3},
    )
)
fig.add_hline(
    y=baseline,
    line_dash="dash",
    line_color="#888888",
    annotation_text=f"random ranking ≈ {baseline:.2f}",
    annotation_position="bottom right",
)
fig.update_layout(
    title="Random Forest — CV PR-AUC by trial",
    xaxis_title="trial",
    yaxis_title="PR-AUC (higher is better)",
    template="plotly_white",
    height=400,
    legend={"orientation": "h", "y": -0.25, "x": 0.5, "xanchor": "center"},
    margin={"b": 90, "t": 50},
)
fig.show()


## 5) XGBoost — TPE vs random

Same three Optuna steps. Same XGBoost knobs both times. Only the **sampler** changes.

| knob | meaning | range |
|------|---------|-------|
| `n_estimators` | how many boosting rounds | 100–400 |
| `max_depth` | depth of each small tree | 2–8 |
| `learning_rate` | step size per new tree (`log=True`) | 0.01–0.2 |

`scale_pos_weight ≈ (# non-defaults) / (# defaults)` is **passed in**, not tuned — we already know the imbalance.

**TPE vs random (the only comparison in this notebook):**

- **TPE** remembers. After a few random starts it samples more where scores were high.
- **Random** has no memory. Trial 20 is still a draw from the full range.

If TPE's best is trial 19 of 20, we did **not** stop early. Raise `N_TRIALS` if you want more tries — more is usually better, with diminishing returns.

### Step A — one objective, reused for both samplers


In [26]:
spw = scale_pos_weight(y_train)
print("scale_pos_weight (fixed) =", round(spw, 2))


def xgb_objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 100, 400, step=50)
    max_depth = trial.suggest_int("max_depth", 2, 8)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.2, log=True)
    model = make_xgboost(
        y_train=y_train,
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        scale_pos_weight=spw,
    )
    return cv_pr_auc(model, x_train, y_train)


scale_pos_weight (fixed) = 3.52


### Steps B and C — TPE study, then run trials


In [27]:
xgb_tpe = optuna.create_study(
    direction="maximize",
    study_name="xgboost_tpe",
    sampler=optuna.samplers.TPESampler(seed=config.RANDOM_STATE, n_startup_trials=5),
)
xgb_tpe.optimize(xgb_objective, n_trials=N_TRIALS)

print("TPE best trial:", xgb_tpe.best_trial.number)
print("TPE best params:", xgb_tpe.best_trial.params)
print("TPE best CV PR-AUC:", round(xgb_tpe.best_trial.value, 3))


TPE best trial: 17
TPE best params: {'n_estimators': 150, 'max_depth': 2, 'learning_rate': 0.02311035115421083}
TPE best CV PR-AUC: 0.515


### Same objective, random sampler

New study, same `xgb_objective`, same `N_TRIALS`. Only `RandomSampler` is different.


In [28]:
xgb_random = optuna.create_study(
    direction="maximize",
    study_name="xgboost_random",
    sampler=optuna.samplers.RandomSampler(seed=config.RANDOM_STATE + 7),
)
xgb_random.optimize(xgb_objective, n_trials=N_TRIALS)

print("Random best trial:", xgb_random.best_trial.number)
print("Random best params:", xgb_random.best_trial.params)
print("Random best CV PR-AUC:", round(xgb_random.best_trial.value, 3))
print(
    "TPE",
    round(xgb_tpe.best_value, 3),
    "vs random",
    round(xgb_random.best_value, 3),
)


Random best trial: 17
Random best params: {'n_estimators': 350, 'max_depth': 4, 'learning_rate': 0.010857383710987108}
Random best CV PR-AUC: 0.488
TPE 0.515 vs random 0.488


Left = TPE, right = random. Orange / green = **best so far**. TPE's line often climbs sooner. Random can still luck into a good draw; it just does not aim.


In [29]:
def add_history(fig, study, col, showlegend, best_color):
    numbers = [t.number for t in study.trials]
    scores = [t.value for t in study.trials]
    best = pd.Series(scores).cummax()
    fig.add_trace(
        go.Scatter(
            x=numbers,
            y=scores,
            mode="markers",
            name="this trial",
            marker={"color": "#2F6F8F", "size": 8},
            showlegend=showlegend,
        ),
        row=1,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=numbers,
            y=best,
            mode="lines",
            name="best so far",
            line={"width": 3, "color": best_color},
            showlegend=showlegend,
        ),
        row=1,
        col=col,
    )
    fig.add_hline(y=baseline, line_dash="dash", line_color="#888888", row=1, col=col)


fig = make_subplots(rows=1, cols=2, subplot_titles=("TPE", "Random"))
add_history(fig, xgb_tpe, 1, True, "#C45C26")
add_history(fig, xgb_random, 2, False, "#1B4332")
fig.update_xaxes(title_text="trial")
fig.update_yaxes(title_text="PR-AUC")
fig.update_layout(
    template="plotly_white",
    height=400,
    legend={"orientation": "h", "y": -0.25, "x": 0.5, "xanchor": "center"},
    margin={"b": 90, "t": 50},
)
fig.show()


## 6) Refit the winners on all of train

Optuna's number is **CV on train**. Now take `best_trial.params`, fit once on the full train set, and score the **hold-out** test set. Watch `pr_auc`, not accuracy.


In [30]:
rf_model = make_random_forest(**rf_study.best_trial.params)
rf_model.fit(x_train, y_train)
rf_holdout = classification_metrics(y_test, predict_proba_positive(rf_model, x_test))

xgb_params = dict(xgb_tpe.best_trial.params)
xgb_params["scale_pos_weight"] = spw
xgb_model = make_xgboost(y_train=y_train, **xgb_params)
xgb_model.fit(x_train, y_train)
xgb_holdout = classification_metrics(y_test, predict_proba_positive(xgb_model, x_test))

summary = pd.DataFrame(
    {
        "random_forest": {"cv_pr_auc": round(rf_study.best_value, 3), **rf_holdout},
        "xgboost": {"cv_pr_auc": round(xgb_tpe.best_value, 3), **xgb_holdout},
    }
).T
display(summary)
print(
    f"Never-default accuracy ≈ {dummy_acc:.1%}. "
    f"Random ranking PR-AUC ≈ {baseline:.2f}."
)
print("Hold-out PR-AUC winner:", summary["pr_auc"].idxmax())


,cv_pr_auc,pr_auc,roc_auc,f1,precision,recall
random_forest,0.496,0.52,0.79,0.56,0.53,0.59
xgboost,0.515,0.54,0.78,0.54,0.49,0.60


Never-default accuracy ≈ 78.0%. Random ranking PR-AUC ≈ 0.22.
Hold-out PR-AUC winner: xgboost


## Takeaways

1. **Objective → create study → `study.optimize` → `study.best_trial`.** Creating the study trains nothing. `optimize` fills the study in place.
2. **TPE remembers; random does not.** We compared them once, on XGBoost, with the same objective.
3. Best at trial 19 of 20 is **not** early stopping. We ran all 20. More trials can still help.
4. Score **PR-AUC** (~0.22 random ranking here), not accuracy (~78% if you never predict default).
5. Full-data training in this repo: `from credit_fraud.pipeline import run_training; run_training()`.
